# Figure 1: the final layer, the readout, and the derived causal structure (Sections 1 and 2)

**Paper section.** Section 1 and Section 2 (Derivation) with Figure 1.

**Claim.** In the words of the caption of Figure 1, "a The transformer architecture's final layer and readout, which converts the residual stream into next-token output probabilities", and "b–e Sketches of the derived causal structure with V = 3 tokens."

**Experiment.** None. The panels are drawings. Panels b to d place 72 token positions on a circle of centered logits and apply a random shift along the all-ones vector, a random positive scaling, and a rotation of the orthogonal component. Panel e shows the first-order causal subspace of Equation (5) at three token positions of a random MLP with three input dimensions.

**Saved Outputs.** In `figures/`: `figure1.pdf`.

**How the notebook works.** The first code cell sets the fonts and the output path. The next two cells compute the points of panels b to d and the planes of panel e. The remaining cells define the architecture diagram of panel a and the helpers that draw the three-dimensional panels back to front, and the last cell draws the figure and writes it to `figures/`.

In [ ]:
import math
import os
from itertools import combinations
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import to_rgba
from matplotlib.patches import Circle, FancyArrowPatch, FancyBboxPatch
from matplotlib.textpath import TextPath, text_to_path
from matplotlib.font_manager import FontProperties
from matplotlib import font_manager
from matplotlib.transforms import Affine2D, ScaledTranslation
from mpl_toolkits.mplot3d import proj3d
from mpl_toolkits.mplot3d.art3d import Line3DCollection, Poly3DCollection

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['font.weight'] = 'light'
plt.rcParams['mathtext.fontset'] = 'cm'                # math and function names in Computer Modern, as in the paper
LM_BOLD = os.path.expanduser('~/Library/Fonts/lmroman10-bold.otf')   # Latin Modern Roman Bold, the cmbx10 of LaTeX's
if os.path.exists(LM_BOLD):                                          # bold math, for the bold 1 (matplotlib's own
    font_manager.fontManager.addfont(LM_BOLD)                        # Computer Modern has no bold digits)
plt.rcParams['mathtext.bf'] = 'Latin Modern Roman:bold' if os.path.exists(LM_BOLD) else 'cmb10'
sns.set_context('notebook')

rng = np.random.default_rng(0)
PLOT_DIR = '../figures'
os.makedirs(PLOT_DIR, exist_ok=True)
FIG_PATH = os.path.join(PLOT_DIR, 'figure1.pdf')

## Panels b to d: centered logits, shifts along the all-ones vector, and scalings

In [ ]:
# Logits. With V = 3 tokens, the centered logits lie on the 2D plane 1-perp inside R^3.
N = 72                                        # token positions
r = 2.4                                       # radius of the circle of centered logits
theta = np.linspace(0, 2 * np.pi, N, endpoint=False)

e1 = np.array([1., -1., 0.]) / np.sqrt(2)     # orthonormal basis of 1-perp
e2 = np.array([1., 1., -2.]) / np.sqrt(6)
L_c = r * (np.cos(theta)[:, None] * e1 + np.sin(theta)[:, None] * e2)   # centered logits, (N, 3)
lse = np.log(np.exp(L_c).sum(1))                                       # logsumexp at each position
P = np.exp(L_c - lse[:, None])                                         # next-token probabilities, the output, (N, 3)

# Each token position is colored by its output, a mix of three token colors weighted by the probabilities.
token_rgb = np.array([[31, 119, 180], [214, 39, 40], [44, 160, 44]]) / 255
colors = np.clip(P @ token_rgb, 0, 1)
colors_light = 0.75 * colors + 0.25                                    # the same colors, a little lighter, for the shifts

# Plot frame for the logits: (x, y) are coordinates inside 1-perp, and z is the component along 1.
PHASE = np.radians(-28)                               # turns the plot's basis of 1-perp, which sets where each color
x0, y0 = r * np.cos(theta + PHASE), r * np.sin(theta + PHASE)   # sits on the ring
z_centered = np.zeros(N)
u = rng.uniform(-1, 1, N)                             # a random shift along 1 at each position, about half of them
z_shift = np.sign(u) * (0.4 + 1.1 * np.abs(u))        # down, each at least 0.4 long

# resid_post. A random positive scale at each position (the per-position scaling of the final RMSNorm), 12 to 45
# percent longer or shorter. Of the positions shown in (c), every other one, half are scaled up and half down.
n_s = rng.normal(0, 1, N)
n_s -= np.median(n_s[::2])
s = 1 + np.sign(n_s) * (0.12 + 0.33 * np.tanh(np.abs(n_s)))

## Panel e: the first-order causal subspace of mlp_in at three token positions

In [ ]:
# mlp_in. First-order causal subspace span{v_i} at three token positions of a random toy MLP,
# with v_i = W_in^T (gelu'(z) * W_out^T g_i) as in the mlp_in paragraph (d_model = 3, V = 3).
d, d_mlp = 3, 16
W_in = rng.normal(0, 1, (d_mlp, d)) / np.sqrt(d)
b_in = rng.normal(0, 0.5, d_mlp)
W_out = rng.normal(0, 1, (d, d_mlp)) / np.sqrt(d_mlp)
G0 = rng.normal(0, 1, (d, 3))
g = G0 - G0.mean(1, keepdims=True)                    # centered columns g_i, which span 2 dimensions

erf = np.vectorize(math.erf)
def gelu_prime(z):
    return 0.5 * (1 + erf(z / np.sqrt(2))) + z * np.exp(-z ** 2 / 2) / np.sqrt(2 * np.pi)

def plane_at(x):
    z = W_in @ x + b_in
    v = W_in.T @ (gelu_prime(z)[:, None] * (W_out.T @ g))          # columns are v_1, v_2, v_3
    U_, S_, _ = np.linalg.svd(v)
    return U_[:, 0], U_[:, 1], U_[:, 2], S_                          # basis of span{v_i}, its normal, singular values

X = rng.normal(0, 1, (40, d))
X *= np.sqrt(d) / np.linalg.norm(X, axis=1, keepdims=True)          # post-RMSNorm activations, norm sqrt(d)
cands = [plane_at(x) for x in X]
def min_angle(idx):
    return min(np.arccos(min(1, abs(cands[i][2] @ cands[j][2]))) for i, j in combinations(idx, 2))
# Of 40 random positions, show the three whose planes differ most, so the planes are easy to tell apart.
mlp_planes = [cands[i] for i in max(combinations(range(len(cands)), 3), key=min_angle)]

normals = np.array([p[2] for p in mlp_planes])
print('third singular value (should be ~0):', [f'{p[3][2]:.1e}' for p in mlp_planes])
print('angles between the per-position planes (deg):',
      [round(float(np.degrees(np.arccos(abs(normals[i] @ normals[j])))), 1) for i, j in [(0, 1), (0, 2), (1, 2)]])

## Panel a: the final layer and the readout

In [ ]:
# Panel (a): the final layer and the readout. The hook names and their boxes are
# smaller, the rows are closer together, each labeled arrow runs a little below its row so that its label sits level
# with the boxes, and the Readout is as wide as the Final Layer. The layout is in inches at the paper's
# column width. architecture(ax, k, width) draws it into ax with every font size and line width scaled by k and the
# arrows stretched so that the diagram is width inches wide at that size.
HOOK_FONT = 'cmtt10'                                 # hook names in LaTeX's \texttt font (Computer Modern Typewriter), as
FS_HOOK, FS_OP, FS_GRP = 9.2, 7.2, 7.2               # in the paper. cmtt is small for its size, hence 9.2
PAD, BH, PR = 0.042, 0.11, 0.055                     # box inner pad, box half-height, plus-circle radius (in)
YD, YF, YR = 0.26, 0.72, 1.34                        # attention/MLP row, main row, readout row
GAP = (YF - PR - 0.008) - (YD + BH + 0.008)          # plain arrows match the vertical ones
LM = 0.10                                            # room around an arrow label in its gap, both sides (in)
LX = 0.21                                            # left edge of the Final Layer's leftmost box
LG = 0.040                                           # an arrow label's baseline sits LG above its arrow
AH = 1.72                                            # height of the diagram (in)

def tw(text, size, family):
    fp = FontProperties(family=family, weight='light')
    return TextPath((0, 0), text, size=size, prop=fp).get_extents().width / 72.0

def hw_of(text):
    return 0.5 * tw(text, FS_HOOK, HOOK_FONT) + PAD

def opw(text):                                       # width of an arrow label as drawn
    fp = FontProperties(family='sans-serif', weight='light', size=FS_OP)
    return text_to_path.get_text_width_height_descent(text, fp, ismath='$' in text)[0] / 72.0

def FN(name):                                        # a function name, upright in the math font, as \mathrm in the paper
    return r'$\mathrm{' + name + '}$'

hw_pre, hw_rm, hw_rp = hw_of('resid_pre'), hw_of('resid_mid'), hw_of('resid_post')
hw_rf, hw_lg, hw_out = hw_of('resid_final'), hw_of('logits'), hw_of('output')
hw_ai, hw_ao, hw_mi, hw_mo = hw_of('attn_in'), hw_of('attn_out'), hw_of('mlp_in'), hw_of('mlp_out')
RX = 0.06                                            # the vertical rail of the hook into the readout

def layout(sx):                                      # positions with every arrow sx times its column-width length
    L = {}
    LGA = (max(opw(FN('Attention')), opw(FN('MLP'))) + LM) * sx                  # bottom row
    gap = GAP * sx
    # Final Layer, laid out left to right, then mirrored right to left
    u_pre = u_ai = LX + max(hw_pre, hw_ai)
    u_p1 = u_ai + hw_ai + LGA + hw_ao
    u_rm = max(u_p1 + PR + gap + hw_rm, u_p1 + hw_ao + 0.08 * sx + hw_mi)
    u_p2 = max(u_rm + hw_mi + LGA + hw_mo, u_rm + hw_rm + gap + PR)
    u_rp = u_p2 + PR + (u_rm - hw_rm - u_p1 - PR) + hw_rp
    SL, SR = u_pre - max(hw_pre, hw_ai) - 0.07, u_rp + hw_rp + 0.07
    LGB = max((SR - 0.07 - SL - 2 * (hw_rf + hw_lg + hw_out)) / 3,      # top row, so that the Readout is as wide
              (max(opw(FN('RMSNorm')), opw(FN('Unembedding')), opw(FN('Softmax'))) + LM) * sx)   # as the Final Layer
    L.update(u_pre=u_pre, u_ai=u_ai, u_p1=u_p1, u_rm=u_rm, u_p2=u_p2, u_rp=u_rp, SL=SL, SR=SR, LGB=LGB,
             W=SR + 0.11 + 0.2 * sx)                 # room on the right for the stream from the previous layers
    # Readout, left to right
    L['x_rf'] = SL + LGB + hw_rf
    L['x_lg'] = L['x_rf'] + hw_rf + LGB + hw_lg
    L['x_out'] = L['x_lg'] + hw_lg + LGB + hw_out
    return L

def fit_layout(width):                               # the stretch that makes the diagram width wide
    lo, hi = 0.5, 6.0
    for _ in range(60):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if layout(mid)['W'] < width else (lo, mid)
    return layout(lo)

def architecture(ax, k, width):
    L = fit_layout(width / k)                        # in column-width inches
    W, SL, SR, LGB = L['W'], L['SL'], L['SR'], L['LGB']
    u_pre, u_ai, u_p1, u_rm, u_p2, u_rp = (L[n] for n in ('u_pre', 'u_ai', 'u_p1', 'u_rm', 'u_p2', 'u_rp'))
    x_rf, x_lg, x_out = L['x_rf'], L['x_lg'], L['x_out']
    mx = lambda x: SL + SR - x
    DL = (LG + 0.75 * FS_OP / 72) / 2               # a labeled arrow and its label are centered on their row
    ax.set_xlim(0, W); ax.set_ylim(0, AH); ax.axis('off')
    def box(x, y, text):
        hw = hw_of(text)
        ax.add_patch(FancyBboxPatch((x - hw, y - BH), 2 * hw, 2 * BH, boxstyle='round,pad=0.008,rounding_size=0.035',
                                    fc='white', ec='black', lw=0.7 * k, zorder=3))
        ax.text(x, y, text, ha='center', va='center', fontsize=FS_HOOK * k, family=HOOK_FONT, zorder=4)
    def link(x1, y1, x2, y2, label=None, at=None):  # a labeled arrow runs DL below its row, its label just above it
        if label:
            y1, y2 = y1 - DL, y2 - DL
            ax.text(0.5 * (x1 + x2) if at is None else at, y1 + LG, label, ha='center', va='baseline',
                    fontsize=FS_OP * k, zorder=4)
        ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='-|>', mutation_scale=6.5 * k, lw=0.7 * k,
                                     color='black', shrinkA=0, shrinkB=0, zorder=2))
    def plus(x, y):
        ax.add_patch(Circle((x, y), PR, fc='white', ec='black', lw=0.7 * k, zorder=3))
        ax.text(x, y - 0.004, '+', ha='center', va='center', fontsize=8 * k, zorder=4)
    def group(x0, x1, y0, y1, label):
        ax.add_patch(FancyBboxPatch((x0, y0), x1 - x0, y1 - y0, boxstyle='round,pad=0.01,rounding_size=0.05',
                                    fc='0.965', ec='0.62', lw=0.6 * k, zorder=1))
        ax.text(x0 + 0.01, y1 + 0.025, label, ha='left', va='bottom', fontsize=FS_GRP * k, style='italic',
                color='0.45', zorder=4)
    # Final Layer (bottom, right to left)
    ax.text(W - 0.02, YF - 0.25 * 9.5 / 72, r'${\cdot}\,{\cdot}\,{\cdot}$', ha='right', va='baseline',
            fontsize=9.5 * k)                       # TeX's \cdots, its dots on the arrow's line (the math axis)
    link(W - 0.19, YF, mx(u_pre) + hw_pre + 0.015, YF)
    for u, y, name in [(u_pre, YF, 'resid_pre'), (u_rm, YF, 'resid_mid'), (u_rp, YF, 'resid_post'),
                       (u_ai, YD, 'attn_in'), (u_p1, YD, 'attn_out'), (u_rm, YD, 'mlp_in'), (u_p2, YD, 'mlp_out')]:
        box(mx(u), y, name)
    plus(mx(u_p1), YF); plus(mx(u_p2), YF)
    link(mx(u_pre) - hw_pre, YF, mx(u_p1) + PR, YF)
    link(mx(u_p1) - PR, YF, mx(u_rm) + hw_rm, YF)
    link(mx(u_rm) - hw_rm, YF, mx(u_p2) + PR, YF)
    link(mx(u_p2) - PR, YF, mx(u_rp) + hw_rp, YF)
    link(mx(u_pre), YF - BH - 0.008, mx(u_ai), YD + BH + 0.008)
    ax.text(mx(u_pre) - 0.06, 0.5 * (YF + YD), FN('RMSNorm'), ha='right', va='center', fontsize=FS_OP * k)
    link(mx(u_ai) - hw_ai, YD, mx(u_p1) + hw_ao, YD, FN('Attention'))
    link(mx(u_p1), YD + BH + 0.008, mx(u_p1), YF - PR - 0.008)
    link(mx(u_rm), YF - BH - 0.008, mx(u_rm), YD + BH + 0.008)
    ax.text(mx(u_rm) - 0.06, 0.5 * (YF + YD), FN('RMSNorm'), ha='right', va='center', fontsize=FS_OP * k)
    link(mx(u_rm) - hw_mi, YD, mx(u_p2) + hw_mo, YD, FN('MLP'))
    link(mx(u_p2), YD + BH + 0.008, mx(u_p2), YF - PR - 0.008)
    # the hook out of resid_post, up the left side and into the Readout row, with rounded corners
    CR = 0.035
    arc1 = [(RX + CR + CR * np.cos(a), YF + CR + CR * np.sin(a)) for a in np.linspace(-np.pi / 2, -np.pi, 12)]
    arc2 = [(RX + CR + CR * np.cos(a), YR - DL - CR + CR * np.sin(a)) for a in np.linspace(np.pi, np.pi / 2, 12)]
    xs, ys = zip(*([(mx(u_rp) - hw_rp, YF), (RX + CR, YF)] + arc1 + [(RX, YR - DL - CR)] + arc2))
    ax.plot(xs, ys, color='black', lw=0.55 * k, zorder=2, solid_joinstyle='round')
    link(RX + CR, YR, x_rf - hw_rf, YR, FN('RMSNorm'), at=SL + 0.5 * LGB)   # label over the part inside the Readout
    # Readout (top, left to right)
    box(x_rf, YR, 'resid_final'); box(x_lg, YR, 'logits'); box(x_out, YR, 'output')
    link(x_rf + hw_rf, YR, x_lg - hw_lg, YR, FN('Unembedding'))
    link(x_lg + hw_lg, YR, x_out - hw_out, YR, FN('Softmax'))
    group(SL, SR, YD - BH - 0.10, YF + BH + 0.08, 'Final Layer')
    group(SL, x_out + hw_out + 0.07, YR - BH - 0.10, YR + BH + 0.10, 'Readout')

## Drawing helpers for the three-dimensional panels

In [ ]:
# matplotlib draws 3D artists in zorder, not by depth. So every panel is drawn back to front by hand (painter's
# algorithm): what lies behind a surface is drawn before it, and the surface is opaque enough to hide it.
AZIM, ELEV = -58, 26                                 # every panel is seen from the same direction
LIM, ZRANGE, ZOOM = 3.0, 6.0, 1.77                   # and drawn in the same box, so all panels share one scale
PLANE = 2.0                                          # half side of every gray plane

EX, EY, EZ = np.eye(3)
FRONT = np.array([np.cos(np.radians(AZIM)), np.sin(np.radians(AZIM)), 0])     # horizontal direction toward the viewer
RIGHT = np.cross(EZ, FRONT)                                                    # screen right
TOWARD = np.cos(np.radians(ELEV)) * FRONT + np.sin(np.radians(ELEV)) * EZ      # toward the viewer
UP = np.cross(TOWARD, RIGHT)                                                   # screen up
SQ1 = np.cos(np.radians(15)) * RIGHT + np.sin(np.radians(15)) * FRONT   # edge directions of the gray planes,
SQ2 = np.cross(EZ, SQ1)                                                 # turned 15 degrees from the view
LIGHT = -0.45 * RIGHT + 0.35 * FRONT + 0.82 * EZ                        # one soft light, above, left and in front
LIGHT /= np.linalg.norm(LIGHT)

def lambert(normal, depth=0.28):                    # brightness of a surface facing the viewer, between 1 - depth and 1
    normal = normal if normal @ TOWARD >= 0 else -normal
    return 1 - depth + depth * max(normal @ LIGHT, 0.0)

class Arrow3D(FancyArrowPatch):
    def __init__(self, p0, p1, **kw):
        super().__init__((0, 0), (0, 0), **kw)
        self._verts3d = np.array([p0, p1], dtype=float).T
    def do_3d_projection(self, renderer=None):
        xs, ys, zs = proj3d.proj_transform(*self._verts3d, self.axes.M)
        self.set_positions((xs[0], ys[0]), (xs[1], ys[1]))
        return np.min(zs)

def arrow(ax, p0, p1, color='0.25', lw=2.8, ms=22, zorder=5):
    ax.add_artist(Arrow3D(p0, p1, arrowstyle='-|>', mutation_scale=ms, lw=lw, color=color,
                          shrinkA=0, shrinkB=0, zorder=zorder, clip_on=False))

T0, D, LAYERS = 0.45, 0.1, 36                        # a gray plane fades out linearly from t0 = T0 of its half side to its
                                                     # border, with the start of the fade rounded off from t0 - D to t0 + D

def fade(t, t0=T0):                                  # 1 in the middle of a gray plane, 0 at its border (|t| = 1)
    t, k = np.abs(t), 1 / (1 - t0)
    g = np.where(t < t0 + D, k * np.clip(t - t0 + D, 0, None) ** 2 / (4 * D), k * (t - t0))
    return np.clip(1 - g, 0, 1)

def unfade(c, t0=T0):                                # the distance from the middle at which fade is c, for 0 < c <= 1
    g, k = 1 - c, 1 / (1 - t0)
    return np.where(g < k * D, t0 - D + np.sqrt(4 * D * g / k), t0 + g / k)

def square(ax, half, zorder, center=(0, 0, 0), u=SQ1, v=SQ2, alpha=0.42, t0=T0):
    # A gray plane that fades out toward its border, like a blurred square. Its darkness at (x, y) along its two edges
    # goes with fade(x) * fade(y), which is smooth everywhere, so no crease shows along its diagonals. It is drawn as
    # nested faint layers, the level sets of fade(x) * fade(y) at evenly spaced distances from the middle, each as
    # faint as the drop in darkness to the next, so that its middle is as dark as one plane of this alpha. Each layer is
    # a square with rounded corners. Its gray is shaded by how much it faces the light.
    gray = float(np.clip(0.8 * lambert(np.cross(u, v)), 0, 1))
    lev = np.r_[fade(np.linspace(t0 - D, 1, LAYERS, endpoint=False), t0), 0]
    s = np.linspace(0, 1, 16)[:, None]
    for c, c_next in zip(lev[:-1], lev[1:]):
        xy = np.hstack([unfade(c ** (1 - s), t0), unfade(c ** s, t0)])  # a rounded corner, from one edge to the next
        rim = np.vstack([xy, xy[:, ::-1] * [-1, 1], -xy, xy[:, ::-1] * [1, -1]])     # and the other three
        pts = np.asarray(center) + half * (rim[:, :1] * u + rim[:, 1:] * v)
        a = 1 - (1 - alpha) ** (c - c_next)
        ax.add_collection3d(Poly3DCollection([pts], facecolor=(gray, gray, gray, a), edgecolor='none', zorder=zorder))
    return np.asarray(center) + half * np.array([u + v, u - v, -u - v, -u + v])           # corners

def dots(ax, xs, ys, zs, cols, zorder, size=70):
    ax.scatter(xs, ys, zs, color=np.atleast_2d(cols)[:, :3], s=size, depthshade=False, edgecolors='none', zorder=zorder)

def behind(p, half=PLANE, t0=T0):                   # how much of a point below the horizontal plane at z = 0 shows
    if p[2] >= 0:
        return 1.0
    q = p - p[2] / TOWARD[2] * TOWARD                # where the line of sight from p crosses the plane
    w = np.clip(1 - fade(q @ SQ1 / half, t0) * fade(q @ SQ2 / half, t0) / 0.18, 0, 1)   # 0 where the plane is dark
    return w * w * (3 - 2 * w)                       # hidden behind the plane, shown again across its faint rim

def perp_label(ax, p, base, size, color, ha='left', math='cm', **kw):   # base with a raised perp almost as large as
    # the text. The perp is drawn as \bot, the same Computer Modern glyph as LaTeX's \perp, which matplotlib lacks.
    prop = FontProperties(family='DejaVu Sans', math_fontfamily=math)  # the text
    w = TextPath((0, 0), base, size=size, prop=prop).get_extents().width
    wp = TextPath((0, 0), r'$\bot$', size=0.85 * size, prop=FontProperties(family='DejaVu Sans')).get_extents().width
    x0 = {'left': 0, 'center': -(w + 1 + wp) / 2, 'right': -(w + 1 + wp)}[ha]   # in points: the pair starts at,
                                                                                # is centered on or ends at p
    at = lambda dx, dy: ax.transData + ScaledTranslation(dx / 72, dy / 72, ax.figure.dpi_scale_trans)
    ax.text(*p, base, fontsize=size, color=color, ha='left', transform=at(x0, 0), math_fontfamily=math, **kw)
    ax.text(*p, r'$\bot$', fontsize=0.85 * size, color=color, ha='left', transform=at(x0 + w + 1, 0.32 * size), **kw)

def to_edge(u, half, a=SQ1, b=SQ2):                 # distance from the center to the edge of a gray plane along u
    return half / max(abs(u @ a), abs(u @ b))

def ring(ax, x, y, z, zorder):
    pts = np.stack([x, y, z], 1)
    ax.add_collection3d(Line3DCollection(np.stack([pts, np.roll(pts, -1, 0)], 1), colors=colors, lw=5, zorder=zorder))

def frame(ax, zmid=0.0, elev=ELEV, shift=(0, 0, 0)):   # shift moves the drawing within its box (horizontal, data units)
    ax.set_proj_type('ortho')
    x, y = -np.asarray(shift, dtype=float)[:2]
    ax.set_xlim(x - LIM, x + LIM); ax.set_ylim(y - LIM, y + LIM); ax.set_zlim(zmid - ZRANGE / 2, zmid + ZRANGE / 2)
    ax.set_box_aspect((2 * LIM, 2 * LIM, ZRANGE), zoom=ZOOM)
    ax.view_init(elev=elev, azim=AZIM)
    ax.set_axis_off()

## Figure 1

In [ ]:
TITLE, HEAD, LABEL, T_FS = 26, 28, 18, 24            # panel titles, hook names, text labels in the panels, t
MATH = 21                                            # math labels and function names (Computer Modern runs small)
TEXT = PERP = '0.4'                                  # one gray for the gray labels
KA, A_X0, A_W = HEAD / FS_HOOK, 0.35, 15.3            # diagram scale (its hook names are HEAD), left edge, width (in)
FS_OP = MATH / KA                                    # the diagram's arrow labels as large as the labels in the panels
ROW, HB, GA = 5.8, 0.45, 0.75                        # heights (in): row of panels, hook names below it, gap above it
FIG_H = HB + 0.87 * ROW + GA + KA * AH + 0.1
fig = plt.figure(figsize=(16, FIG_H))
fig.patch.set_facecolor('white')
y = lambda frac: (HB + frac * ROW) / FIG_H           # positions within the row of panels, as figure fractions
axs = [fig.add_axes([k / 4, y(0.02), 1 / 4, 0.72 * ROW / FIG_H], projection='3d', computed_zorder=False)
       for k in range(4)]
ax_a = fig.add_axes([A_X0 / 16, (HB + 0.87 * ROW + GA) / FIG_H, A_W / 16, KA * AH / FIG_H])
architecture(ax_a, KA, A_W)

# (b) logits. The centered logits lie on the plane 1-perp (ring). Adding c times 1 to the logits at a position leaves
# its output unchanged, and c can differ across positions, so each position is shifted along 1 by its own random amount.
# About half of the shifts go below the plane. Every other position is shown, as in (c), back to front. What lies below
# the plane is drawn before it and is hidden where the plane is in front of it. What lies above is drawn after it, the
# shifts behind the arrow 1 before the arrow and the ones in front of it after.
T0_B = 0.7                                           # the plane of (b) fades out only near its border
ax, f = axs[0], 0.67 * PLANE / r                     # f scales the logits so that the ring sits well inside the plane, as
                                                     # in (c), and the plane shows around it
xa, ya, za = f * x0, f * y0, 0.88 * PLANE / r * z_shift   # the shifts along 1 are drawn longer than the ring's scale,
                                                     # so that those below the plane come out from behind it
idx = np.arange(0, N, 2)
depth = np.stack([xa, ya], 1) @ FRONT[:2]            # positive in front of the arrow 1
across = np.stack([xa, ya], 1) @ RIGHT[:2]           # to the right of the arrow 1 on the page
i1 = idx[np.argmin(np.where(depth[idx] < 0, np.abs(across[idx]), np.inf))]
za[i1] = -abs(za[i1])                                # the position right behind the arrow 1 is shifted down, so that its
                                                     # shift is hidden by the plane instead of running along the arrow
order = idx[np.argsort(depth[idx])]
for k, i in enumerate(order):                        # below the plane
    if za[i] < 0:
        zo = 1 + k / len(idx)
        line = np.linspace([xa[i], ya[i], 0], [xa[i], ya[i], za[i]], 41)
        shown = [0.0] + [behind(p, t0=T0_B) for p in line[1:]]   # it comes out from behind the plane partway down
        k0 = next((m for m, a in enumerate(shown) if a >= 0.5), len(line))
        if k0 <= 0.85 * (len(line) - 1):             # drawn only if at least 15% of it comes out, else hidden whole
            ax.plot(*line[k0:].T, color=colors_light[i], lw=2.2, zorder=zo)
            ax.scatter(xa[i], ya[i], za[i], color=colors_light[i], s=70, depthshade=False, edgecolors='none',
                       zorder=zo + 0.5 / len(idx))
square(ax, PLANE, zorder=2, t0=T0_B)
ring(ax, xa, ya, 0 * xa, zorder=3)
for k, i in enumerate(order):                        # above the plane
    if za[i] > 0:
        zo = (4.1 if depth[i] > 0 else 3.1) + 0.8 * k / len(idx)
        ax.plot([xa[i], xa[i]], [ya[i], ya[i]], [0, za[i]], color=colors_light[i], lw=2.2, zorder=zo)   # as in (c)
        dots(ax, xa[i], ya[i], za[i], colors_light[i], zorder=zo + 0.01)
TIP = 2.6 * EZ                                       # the tip of the direction 1, in gray, taller than the shifts
arrow(ax, (0, 0, 0), TIP, color='0.55', zorder=4)
ax.text(*(TIP - 0.2 * RIGHT), r'$\mathbf{1}$', fontsize=MATH, color=TEXT, ha='right', va='center',
        math_fontfamily='custom', zorder=10)
perp_label(ax, 0.9 * PLANE * SQ1 + 0.15 * RIGHT, r'$\mathbf{1}$', MATH, PERP, va='center', math='custom',
           zorder=10)                                # at the middle of the plane's right edge, away from the ring, so
                                                     # that it names the plane
frame(ax, zmid=0.3)                                  # framed again at the end, level with (c)

# (c) resid_post, component in span{g_i}. Each position can move only along its ray from the origin. The ring is the
# unscaled component (the centered logits, up to a linear map), and each scaled position is joined to its point on it.
# Everything here lies in one plane, so (c) is seen from straight above, as a flat 2D picture. Its plane is the same
# size as the others and is drawn square to the page.
ax, up_b = axs[1], -FRONT                            # seen from above, screen up is the horizontal direction away from us
corners = square(ax, PLANE, zorder=0, u=RIGHT, v=up_b)
idx = np.arange(0, N, 2)                             # every other position, each on its own ray
rays = np.stack([x0[idx], y0[idx], 0 * idx], 1) / r
j = np.argmax(P[idx, 2])                             # the position t shown in (c) and (d), the one most likely to give
                                                     # the green token. It keeps its unscaled length
s_b = s[idx].copy()
s_b[j] = 1.0
a_t = rays[j]                                        # the direction of t's arrow, and the side of it that is up on the page
n_t = np.cross(EZ, a_t) * np.sign(np.cross(EZ, a_t) @ up_b)
T_AT = 0.19 * a_t + 0.23 * n_t                      # the label t, just above the base of its arrow (in ring radii)
pts_t = s_b[:, None] * rays
to_arrow = np.linalg.norm(pts_t - np.clip(pts_t @ a_t, 0, 1)[:, None] * a_t, axis=1)
near = ((np.linalg.norm(pts_t - T_AT, axis=1) < 0.5) | (to_arrow < 0.2)) & (s_b < 1)
s_b[near] = 2 - s_b[near]                           # the points scaled down near the label or the arrow are scaled up
up = np.flatnonzero((s_b > 1) & ~near)              # instead, to leave room, and as many of the points scaled up the least
s_b[up[np.argsort(s_b[up])[:near.sum()]]] = 2 - s_b[up[np.argsort(s_b[up])[:near.sum()]]]   # are scaled down instead
f = min(0.95 * PLANE / r, min(0.93 * to_edge(u, PLANE, RIGHT, up_b) / (s_ * r) for u, s_ in zip(rays, s_b)))
segs, cols = [], []                                 # thin rays from the center that fade out with the plane
for u in rays:
    end = to_edge(u, PLANE, RIGHT, up_b)
    for t0 in np.linspace(0, end, 41)[:-1]:
        mid = (t0 + end / 80) * u
        w = np.clip(fade(mid @ RIGHT / PLANE) * fade(mid @ up_b / PLANE) / 0.64, 0, 1)
        segs.append([t0 * u, (t0 + end / 40) * u])
        cols.append((0.62, 0.62, 0.62, 0.6 * w * w * (3 - 2 * w)))
ax.add_collection3d(Line3DCollection(segs, colors=cols, lw=0.8, zorder=1))
for u, s_, c in zip(rays, s_b, colors_light[idx]):
    ax.plot(*np.array([f * r * u, f * r * s_ * u]).T, color=c, lw=2.2, zorder=1.5)       # as in (b)
ring(ax, f * x0, f * y0, z_centered, zorder=2)
pts_c = f * r * s_b[:, None] * rays                 # the scaled positions
dots(ax, *pts_c.T, colors_light[idx], zorder=3)
pos_c, col_c = f * r * rays[j], colors[idx][j]      # its vector from the origin, ending on the ring
arrow(ax, (0, 0, 0), pos_c, color=col_c, lw=4, ms=24, zorder=4)
ax.text(*(f * r * T_AT), r'$t$', fontsize=T_FS, color=col_c, ha='center', va='center', zorder=10)
anchor_c = corners[np.argmin(corners @ (RIGHT + up_b))] - 0.68 * up_b
ax.text(*anchor_c, r'$\mathrm{span}\{g_i\}$', fontsize=MATH, color=PERP, ha='left', va='baseline',
        zorder=10)                                   # below the plane's lower-left corner, as in (d), framed after (d)

# (d) resid_post, one token position. The colored arrow is the position's component in span{g_i}, and the gray plane
# is span{g_i}-perp (two of its directions). With the span{g_i} component fixed, the span{g_i}-perp component can point
# in any direction (magenta, all on the dotted ring) and the output stays the same.
ax = axs[2]
c_pos = '#d01c8b'                                    # magenta: the same output
az = np.radians(AZIM)
def flat(phi):                                       # unit vector in span{g_i}-perp at angle phi
    return np.array([np.cos(phi), np.sin(phi), 0])
B_ = 1.75                                            # length of the span{g_i}-perp components
H_ = np.linalg.norm(pos_c) / np.cos(np.radians(ELEV))   # the span{g_i} component, as long on the page as t's
                                                     # arrow in (c): a vertical arrow is drawn cos(ELEV) shorter
top = H_ * EZ
corners = square(ax, PLANE, zorder=0)                # span{g_i}-perp, centered under the arrow
arrow(ax, (0, 0, 0), top, color=col_c, lw=4, ms=28, zorder=1)   # the span{g_i} component: the colored arrow of (c)
ph = np.linspace(0, 2 * np.pi, 721)                  # a dotted ring: dots evenly spaced as seen on the page
loop = top + B_ * np.stack([np.cos(ph), np.sin(ph), 0 * ph], 1)
arc = np.r_[0, np.cumsum(np.linalg.norm(np.diff(np.stack([loop @ RIGHT, loop @ UP], 1), axis=0), axis=1))]
at = np.interp(np.arange(int(arc[-1] / 0.15)) * arc[-1] / int(arc[-1] / 0.15), arc, ph)
ax.scatter(*(top + B_ * np.stack([np.cos(at), np.sin(at), 0 * at], 1)).T, color=c_pos, s=18, depthshade=False,
           edgecolors='none', zorder=2)
for dphi in (-60, 60, 150):                          # span{g_i}-perp components of the same length, different directions.
    tip = top + B_ * flat(az + np.radians(dphi))     # The leftmost is solid, and the other two are dotted like the ring,
    d = tip - top                                    # with a solid head
    if dphi == -60:
        arrow(ax, top, tip, color=c_pos, lw=4, ms=28, zorder=3)
        perp_label(ax, top + 0.5 * d + 0.27 * UP, r'$t$', T_FS, c_pos, ha='center', va='center',
                   zorder=10)                        # labeled t-perp, above it and under the dotted ring
        continue
    n_dots = int(np.hypot(d @ RIGHT, d @ UP) / 0.15)
    shaft = top + np.linspace(0, 0.8, max(n_dots, 2))[:, None] * d
    ax.scatter(*shaft.T, color=c_pos, s=18, depthshade=False, edgecolors='none', zorder=3)
    arrow(ax, tip - 0.25 * d / np.linalg.norm(d), tip, color=c_pos, lw=4, ms=28, zorder=3)
ax.text(*(0.28 * EZ + 0.31 * RIGHT), r'$t$', fontsize=T_FS, color=col_c, ha='center', va='center',
        zorder=10)                                  # just right of the base of its arrow, as in (c)
LIFT_D = 0.15                                        # the drawing sits LIFT_D higher on the page, and its span label stays
anchor_d = corners[np.argmin(corners @ RIGHT)] - (0.68 + LIFT_D) * UP
perp_label(ax, anchor_d, r'$\mathrm{span}\{g_i\}$', MATH, PERP, va='baseline',
           zorder=10)                                # below the plane's lower-left corner
ax.text(*(top + (B_ * np.sin(np.radians(ELEV)) + 0.25) * UP), 'same output', fontsize=LABEL, color=c_pos, ha='center',
        va='bottom', zorder=10)                      # just above the ring
zmid_d = 1.15 - LIFT_D / np.cos(np.radians(ELEV))
frame(ax, zmid=zmid_d)

# (c) is framed here, raised so that its span{g_i} lines up with the span{g_i}-perp of (d). Both labels hang from their
# anchors and all panels share one scale, so the anchors must sit equally high above the middle of their boxes. It is
# seen from 89.99 degrees, not 90: straight down, matplotlib's view direction is set by rounding noise and turns the
# drawing by about a degree on the page.
frame(axs[1], elev=89.99, shift=((anchor_d - zmid_d * EZ) @ UP - anchor_c @ up_b) * up_b)

# (e) mlp_in. First-order causal subspace C_t at three token positions t of a random toy MLP, inside the mlp_in space
# (gray floor and back wall). Each square is turned within its own plane, which leaves the subspace unchanged, so that
# all three fit in a small box. Each square is then cut into four wedges along its intersections with the other two,
# so that the wedges can be drawn back to front, and shaded by how much it faces the light.
ax = axs[3]
pcols = [plt.cm.plasma(t) for t in (0.1, 0.5, 0.8)]
BOX = np.stack([SQ1, SQ2, EZ], 1)                    # axes of the box: the edges of the gray planes, and vertical
def fit(a, b):                                       # in-plane basis whose square needs the smallest box
    turns = [(np.cos(t) * a + np.sin(t) * b, -np.sin(t) * a + np.cos(t) * b) for t in np.radians(np.arange(0, 90, 0.5))]
    return min(turns, key=lambda uv: abs(np.array([uv[0] + uv[1], uv[0] - uv[1]]) @ BOX).max())
squares = [fit(a, b) + (n,) for a, b, n, _ in mlp_planes]
HALF = 0.74 * PLANE                                  # half side of each square, small enough that (e) and its key fit
                                                     # in the height of the other drawings
C_ = 1.03 * HALF                                     # half side of the box, which holds the three squares
square(ax, C_, zorder=0, center=-C_ * EZ, alpha=0.42)                            # floor, turned like the gray planes
square(ax, C_, zorder=0.1, center=C_ * SQ2, u=SQ1, v=EZ, alpha=0.31)             # back wall
wedges = []
for k, (a, b, n) in enumerate(squares):
    cuts = np.sort([np.arctan2(c @ b, c @ a) % np.pi + m * np.pi
                    for j in range(3) if j != k for c in [np.cross(n, squares[j][2])] for m in (0, 1)])
    for lo, hi in zip(cuts, np.r_[cuts[1:], cuts[0] + 2 * np.pi]):
        ang = np.array([lo, *[c for c in np.pi / 4 + np.pi / 2 * np.arange(12) if lo < c < hi], hi])
        rad = HALF / np.maximum(abs(np.cos(ang)), abs(np.sin(ang)))
        edge = (rad * np.cos(ang))[:, None] * a + (rad * np.sin(ang))[:, None] * b
        wedges.append((k, np.vstack([np.zeros(3), edge])))
wedges.sort(key=lambda w: w[1].mean(0) @ TOWARD)
for zo, (k, wedge) in enumerate(wedges):
    lit = np.clip(np.array(pcols[k][:3]) * lambert(squares[k][2], depth=0.15), 0, 1)
    ax.add_collection3d(Poly3DCollection([wedge], facecolor=to_rgba(lit, 0.5), edgecolor='none', zorder=1 + zo))
KEY_Y = y(0.125)                                     # baseline of the key, the line the span labels of (c), (d) sit on
for k in range(3):                                   # a key in one row below the drawing
    fig.text(0.875 + 0.055 * (k - 1), KEY_Y, f'$C_{{t={k + 1}}}$', color=pcols[k], fontsize=MATH, ha='center',
             va='baseline')
frame(ax, zmid=-0.4)

letters = 'bcde'
titles = [r'Shift along $\mathbf{1}$' + '\nper position', 'Random scaling\nper position',
          'Rotation at\none position', 'Causal subspace\nper position']
def width(text, size, bold=False):                   # width of a line of text on the page, as a fraction of the figure
    prop = FontProperties(family='DejaVu Sans', weight='bold' if bold else 'normal', math_fontfamily='custom')
    return TextPath((0, 0), text, size=size, prop=prop).get_extents().width / (72 * fig.get_figwidth())
for k, t in enumerate(titles):                       # each letter and title centered over its panel
    block = 0.028 + max(width(line, TITLE) for line in t.split('\n'))
    x = (k + 0.5) / 4 - block / 2
    fig.text(x, y(0.87), letters[k], fontsize=TITLE, fontweight='bold', ha='left', va='top')
    fig.text(x + 0.028, y(0.87), t, fontsize=TITLE, ha='left', va='top', linespacing=1.1, math_fontfamily='custom')
fig.text(0.005, 1 - 0.1 / FIG_H, 'a', fontsize=TITLE, fontweight='bold', ha='left', va='top')
for x, name in [(1 / 8, 'logits'), (1 / 2, 'resid_post'), (7 / 8, 'mlp_in')]:     # centered under their panels, in
    hw, hh = hw_of(name) * KA, BH * KA               # boxes like the hooks of (a)
    cy = HB + 0.07 * ROW - hh - 0.008 * KA           # box center (in), just below the drawings
    fig.add_artist(FancyBboxPatch((16 * x - hw, cy - hh), 2 * hw, 2 * hh,   # in inches, kept with the figure when cropped
                                  transform=Affine2D().scale(1 / 16, 1 / FIG_H) + fig.transFigure,
                                  boxstyle=f'round,pad={0.008 * KA},rounding_size={0.035 * KA}', fc='white', ec='black',
                                  lw=0.7 * KA))
    fig.text(x, cy / FIG_H, name, fontsize=HEAD, family=HOOK_FONT, ha='center', va='center')

# The span labels of (c) and (d) sit on one line with the key of (e): each hangs from its anchor by its baseline, and
# the drawings share one scale, so (d) is moved up by the gap between its anchor and the key's on the page, and (c)
# follows (d) as above.
fig.canvas.draw()
page_y = lambda ax, p: ax.transData.transform(proj3d.proj_transform(*p, ax.M)[:2])[1]   # pixels from the bottom
per_unit = page_y(axs[2], UP) - page_y(axs[2], 0 * UP)                  # pixels per data unit, the same in every panel
rise = (fig.transFigure.transform((0, KEY_Y))[1] - page_y(axs[2], anchor_d)) / per_unit
zmid_d -= rise / np.cos(np.radians(ELEV))
frame(axs[2], zmid=zmid_d)
lift_c = (anchor_d - zmid_d * EZ) @ UP - anchor_c @ up_b
frame(axs[1], elev=89.99, shift=lift_c * up_b)
# (b) is moved so that the tip of 1 is a little below the top edge of the plane of (c), clear of its title: the
# panels share one scale and their boxes are centered at the same height, so the heights above the middles of the
# boxes are compared directly.
frame(axs[0], zmid=TIP @ EZ - (PLANE + lift_c - 0.2) / np.cos(np.radians(ELEV)))

fig.savefig(FIG_PATH, bbox_inches='tight')
plt.show()